# 第 9 章: 特徴量エンジニアリングの探索と可視化

標準化の前後の分布、特徴量と価格の関係、価格の外れ値、天気ごとの利用者数を確認する。

Polyglot Notebooks（.NET Interactive）は 2026 年に廃止された。この Notebook は `Microsoft.dotnet-interactive` 1.0.712001 と Plotly.NET.Interactive 5.0.0 で動作を確かめている。
先に `dotnet build` で `apps/fsharp/` のライブラリをビルドしておく。

In [ ]:
#r "nuget: FSharp.Data, 8.2.0"
#r "nuget: Microsoft.ML, 5.0.0"
#r "nuget: Plotly.NET, 5.1.0"
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "../src/MachineLearning/bin/Debug/net10.0/MachineLearning.dll"

In [ ]:
open System.IO
open Plotly.NET
open MachineLearning.Dataset
open MachineLearning.Chapter09.BikeWeather
open MachineLearning.Chapter09.BostonFeatures
open MachineLearning.Chapter09.Outliers
open MachineLearning.Chapter09.Standardizer

let _, split = prepareBoston (Path.Combine(dataDir (), "Boston.csv")) 0.3 0

## 標準化の前後で分布を比べる

In [ ]:
let standardized = split.XTrain |> transformStandardized (fitStandardizer [ "RM" ] split.XTrain)

Chart.Point(x = (split.XTrain |> List.map (fun row -> row["RM"])), y = (standardized |> List.map (fun row -> row["RM"])))
|> Chart.withTitle "RM の標準化の前後"
|> Chart.withXAxisStyle "標準化前"
|> Chart.withYAxisStyle "標準化後"

## 特徴量と価格の関係を見る

In [ ]:
[ "RM"; "LSTAT" ]
|> List.map (fun column ->
    Chart.Point(x = (split.XTrain |> List.map (fun row -> row[column])), y = split.TTrain, Name = column)
    |> Chart.withXAxisStyle column
    |> Chart.withYAxisStyle "PRICE")
|> Chart.Grid(1, 2)
|> Chart.withTitle "訓練データの特徴量と PRICE"

## 外れ値を見る

In [ ]:
let flags = iqrOutliers split.TTrain
let ranked = List.zip split.TTrain flags |> List.sortBy fst |> List.indexed

[ false; true ]
|> List.map (fun outlier ->
    let points = ranked |> List.filter (fun (_, (_, flag)) -> flag = outlier)
    Chart.Point(x = (points |> List.map fst), y = (points |> List.map (snd >> fst)), Name = (if outlier then "外れ値" else "それ以外")))
|> Chart.combine
|> Chart.withTitle "訓練データの PRICE（小さい順）"
|> Chart.withXAxisStyle "順位"
|> Chart.withYAxisStyle "PRICE"

In [ ]:
[| {| 第1四分位点 = quantile split.TTrain 0.25; 第3四分位点 = quantile split.TTrain 0.75; 外れ値の数 = flags |> List.filter id |> List.length |} |]

## 天気ごとの利用者数を見る

In [ ]:
let means =
    loadBike (Path.Combine(dataDir (), "bike.tsv"))
    |> joinWeather (loadWeather (Path.Combine(dataDir (), "weather.csv")))
    |> meanCountByWeather

Chart.Column(values = (means |> List.map snd), Keys = (means |> List.map fst))
|> Chart.withTitle "天気ごとの平均利用者数"
|> Chart.withYAxisStyle "平均利用者数"